<a href="https://colab.research.google.com/github/solive-11/dissertation-weed-detection/blob/main/notebooks/03_dataset_split_and_experiment_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Environment and reproducibility
# ============================================================

import os
import sys
import platform
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation_weed_detection")

DATA_DIR = PROJECT_ROOT / "01_data"
ENV_DIR = PROJECT_ROOT / "environment"
CONFIG_DIR = PROJECT_ROOT / "configs"
LOG_DIR = PROJECT_ROOT / "logs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"

SEED = 42

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

print("=" * 70)
print("ENVIRONMENT & REPRODUCIBILITY")
print("=" * 70)

print(f"Python        : {sys.version}")
print(f"Platform      : {platform.platform()}")
print(f"PyTorch       : ", end="")

try:
    import torch
    print(torch.__version__)
except ImportError:
    print("NOT INSTALLED")

print(f"NumPy         : {np.__version__}")
print(f"Pandas        : {pd.__version__}")

try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA version  : {torch.version.cuda}")
        print(f"GPU           : {torch.cuda.get_device_name(0)}")
        print(
            f"GPU count     : "
            f"{torch.cuda.device_count()}"
        )

except Exception as e:
    print(f"GPU check error: {e}")

print()
print(f"Random seed   : {SEED}")
print(f"Project root  : {PROJECT_ROOT}")

ENVIRONMENT & REPRODUCIBILITY
Python        : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Platform      : Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch       : 2.11.0+cpu
NumPy         : 2.1.3
Pandas        : 2.2.3
CUDA available: False

Random seed   : 42
Project root  : /content/drive/MyDrive/dissertation_weed_detection


In [3]:
# Project directory structure

directories = [
    DATA_DIR,
    DATA_DIR / "processed",
    DATA_DIR / "splits",
    ENV_DIR,
    CONFIG_DIR,
    LOG_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,]

for directory in directories:
    directory.mkdir(
        parents=True,
        exist_ok=True)

print("PROJECT DIRECTORIES")
print("=" * 70)

for directory in directories:
    print(
        f"{directory}: "
        f"{'✓ exists' if directory.exists() else '✗ missing'}")

PROJECT DIRECTORIES
/content/drive/MyDrive/dissertation_weed_detection/01_data: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/01_data/processed: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/01_data/splits: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/environment: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/configs: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/logs: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/checkpoints: ✓ exists
/content/drive/MyDrive/dissertation_weed_detection/results: ✓ exists


In [4]:
# Load prepared dataset manifest

PROCESSED_DIR = PROJECT_ROOT / "01_data" / "processed"

# Expected manifests from Notebook 02
candidate_manifests = [
    PROCESSED_DIR / "dataset_manifest_with_groups.csv",
    PROCESSED_DIR / "dataset_manifest.csv",
    ]
print("=" * 70)
print("SEARCHING FOR PREPARED MANIFEST")
print("=" * 70)

manifest_path = None
for path in candidate_manifests:
    print(f"{path.name:40} : {'FOUND' if path.exists() else 'NOT FOUND'}")
    if path.exists():
        manifest_path = path
        break

if manifest_path is None:
    raise FileNotFoundError(
        "Neither dataset_manifest_with_groups.csv nor dataset_manifest.csv was found. "
        "Check the processed directory before continuing.")

df = pd.read_csv(manifest_path)

print()
print(f"Loaded manifest : {manifest_path}")
print(f"Rows            : {len(df):,}")
print(f"Columns         : {len(df.columns)}")

print()
print("Columns:")
for col in df.columns:
    print(f"  - {col}")

SEARCHING FOR PREPARED MANIFEST
dataset_manifest_with_groups.csv         : FOUND

Loaded manifest : /content/drive/MyDrive/dissertation_weed_detection/01_data/processed/dataset_manifest_with_groups.csv
Rows            : 6,656
Columns         : 5

Columns:
  - image_id
  - image_path
  - annotation_path
  - image_filename
  - near_duplicate_group


In [5]:
# ============================================================
# Manifest structure verification
# ============================================================

print("=" * 70)
print("MANIFEST STRUCTURE")
print("=" * 70)

display(df.head())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate image IDs:")
print(df["image_id"].duplicated().sum())

print("\nDuplicate filenames:")
print(df["image_filename"].duplicated().sum())

MANIFEST STRUCTURE


,image_id,image_path,annotation_path,image_filename,near_duplicate_group
0,090800iy3emo472414_684,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,090800iy3emo472414_684.jpeg,NDG_00001
1,090805j6869f452411_137,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,090805j6869f452411_137.jpeg,NDG_00002
2,090807o0g127452411_140,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,090807o0g127452411_140.jpeg,NDG_00003
3,09080gy3876a462413_765,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,09080gy3876a462413_765.jpeg,NDG_00004
4,09080rlexxy5462413_146,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,09080rlexxy5462413_146.jpeg,NDG_00005



Data types:


,0
image_id,object
image_path,object
annotation_path,object
image_filename,object
near_duplicate_group,object



Missing values:


,0
image_id,0
image_path,0
annotation_path,0
image_filename,0
near_duplicate_group,0



Duplicate image IDs:
0

Duplicate filenames:
0


In [6]:
# ============================================================
# Cell 5: Near-duplicate group verification
# ============================================================

GROUP_COLUMN = "near_duplicate_group"

if GROUP_COLUMN not in df.columns:
    raise KeyError(
        f"Expected column '{GROUP_COLUMN}' was not found.\n"
        f"Available columns: {list(df.columns)}"
    )

print("=" * 70)
print("NEAR-DUPLICATE GROUP VERIFICATION")
print("=" * 70)

print(f"Total images          : {len(df):,}")
print(f"Unique group IDs      : {df[GROUP_COLUMN].nunique():,}")
print(
    f"Images in multi-image "
    f"groups                 : "
    f"{(df.groupby(GROUP_COLUMN).size() > 1).sum():,}"
)

group_sizes = df.groupby(GROUP_COLUMN).size()

print()
print("Group-size distribution:")
display(group_sizes.value_counts().sort_index())

NEAR-DUPLICATE GROUP VERIFICATION
Total images          : 6,656
Unique group IDs      : 6,439
Images in multi-image groups                 : 134

Group-size distribution:


,count
1,6305
2,98
3,22
4,3
5,2
6,2
7,4
8,2
11,1


In [9]:
# ============================================================
# Class distribution across near-duplicate groups
# ============================================================

import json
from collections import defaultdict

print("=" * 70)
print("CLASS DISTRIBUTION ACROSS NEAR-DUPLICATE GROUPS")
print("=" * 70)

# ------------------------------------------------------------
# Official detection class names
# ------------------------------------------------------------

class_names = {
    0: "kena_commplina_benghalensio",
    1: "lavhala_cyperus_rotundus",
    2: "lambs_quarter_plant",
    3: "Little_mallow",
    4: "moti_dudhi_euphorbia_geneculata_L",
    5: "Obscure_morning _glory",
    6: "Asian_pigeonwings",
    7: "bilayat_mexicana_argemone",
    8: "choti_dudhi_Euphorbia_hirta",
    9: "Digitaria_sp",
    10: "gajar_gavat_congressgavat",
    11: "Graceful_sandmat",
    12: "Sicklepod",
    13: "harali_cynodon_dactylon",
    14: "dwarf_cassia"
}

# ------------------------------------------------------------
# Verify annotation column
# ------------------------------------------------------------

annotation_column = "annotation_path"

if annotation_column not in df.columns:
    raise KeyError(
        f"'{annotation_column}' not found.\n"
        f"Available columns: {list(df.columns)}"
    )

print(f"Annotation column: {annotation_column}")

# ------------------------------------------------------------
# Extract classes for every image
# ------------------------------------------------------------

image_classes = {}

for _, row in df.iterrows():

    annotation_path = Path(row[annotation_column])

    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Annotation not found:\n{annotation_path}"
        )

    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    # --------------------------------------------------------
    # The JSON annotation is a LIST directly.
    # Each element contains a class_id.
    # --------------------------------------------------------

    if not isinstance(annotation, list):
        raise TypeError(
            f"Unexpected JSON structure in {annotation_path}\n"
            f"Expected list, got {type(annotation)}"
        )

    classes = set()

    for obj in annotation:

        if "class_id" not in obj:
            raise KeyError(
                f"'class_id' missing in annotation:\n"
                f"{annotation_path}"
            )

        classes.add(int(obj["class_id"]))

    image_classes[row["image_id"]] = classes

print(f"Images processed: {len(image_classes):,}")

# ------------------------------------------------------------
# Aggregate classes by near-duplicate group
# ------------------------------------------------------------

group_classes = defaultdict(set)

for _, row in df.iterrows():

    image_id = row["image_id"]
    group_id = row[GROUP_COLUMN]

    group_classes[group_id].update(
        image_classes[image_id]
    )

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print()
print(
    f"{'Class':<8}"
    f"{'Images':>10}"
    f"{'Groups':>10}"
    f"{'Group %':>12}"
)

print("-" * 42)

total_groups = df[GROUP_COLUMN].nunique()

for class_id in sorted(class_names):

    image_count = sum(
        class_id in classes
        for classes in image_classes.values()
    )

    group_count = sum(
        class_id in classes
        for classes in group_classes.values()
    )

    percentage = (
        100 * group_count / total_groups
    )

    print(
        f"{class_id:<8}"
        f"{image_count:>10,}"
        f"{group_count:>10,}"
        f"{percentage:>11.2f}%"
    )

CLASS DISTRIBUTION ACROSS NEAR-DUPLICATE GROUPS
Annotation column: annotation_path
Images processed: 6,656

Class       Images    Groups     Group %
------------------------------------------
0            3,377     3,248      50.44%
1            5,059     4,898      76.07%
2            5,546     5,377      83.51%
3            1,356     1,293      20.08%
4            1,021       972      15.10%
5            1,091     1,063      16.51%
6              173       164       2.55%
7              186       180       2.80%
8              615       592       9.19%
9              721       707      10.98%
10             343       336       5.22%
11              88        85       1.32%
12             333       315       4.89%
13              55        55       0.85%
14             228       223       3.46%


In [10]:
# ============================================================
# Leakage-aware group split
# ============================================================

from collections import defaultdict
import numpy as np
import pandas as pd

print("=" * 70)
print("CONSTRUCTING LEAKAGE-AWARE TRAIN / VAL / TEST SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# Split configuration
# ------------------------------------------------------------

SPLIT_RATIOS = {
    "train": 0.80,
    "val": 0.10,
    "test": 0.10
}

SEED = 42

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# Get unique groups
# ------------------------------------------------------------

unique_groups = list(df[GROUP_COLUMN].unique())

print(f"Total groups : {len(unique_groups):,}")

# ------------------------------------------------------------
# Target number of groups
# ------------------------------------------------------------

n_groups = len(unique_groups)

target_groups = {
    split: round(n_groups * ratio)
    for split, ratio in SPLIT_RATIOS.items()
}

# Correct rounding so total = n_groups
difference = n_groups - sum(target_groups.values())

target_groups["train"] += difference

print()
print("Target groups:")
for split, count in target_groups.items():
    print(f"  {split:<10}: {count:,}")

# ------------------------------------------------------------
# Build class -> groups mapping
# ------------------------------------------------------------

class_to_groups = defaultdict(set)

for group_id, classes in group_classes.items():

    for class_id in classes:
        class_to_groups[class_id].add(group_id)

# ------------------------------------------------------------
# Greedy stratified assignment
#
# Rare classes are processed first.
# Each group is assigned to the split where it
# contributes most toward the desired class distribution.
# ------------------------------------------------------------

group_list = list(unique_groups)

# Shuffle deterministically before sorting
rng.shuffle(group_list)

# Process groups containing rare classes first
group_list.sort(
    key=lambda g: (
        -sum(
            1 / max(len(class_to_groups[c]), 1)
            for c in group_classes[g]
        ),
        -len(group_classes[g])
    )
)

assignments = {}

split_group_counts = {
    "train": 0,
    "val": 0,
    "test": 0
}

# Desired class counts by split
class_group_targets = {}

for class_id in class_names:

    total = len(class_to_groups[class_id])

    class_group_targets[class_id] = {
        split: total * ratio
        for split, ratio in SPLIT_RATIOS.items()
    }

class_group_current = {
    split: Counter()
    for split in SPLIT_RATIOS
}

# ------------------------------------------------------------
# Assign groups
# ------------------------------------------------------------

for group_id in group_list:

    classes = group_classes[group_id]

    available_splits = [
        split
        for split in SPLIT_RATIOS
        if split_group_counts[split] < target_groups[split]
    ]

    if not available_splits:
        raise RuntimeError(
            "No available split while groups remain."
        )

    # Score each possible split
    scores = {}

    for split in available_splits:

        score = 0.0

        for class_id in classes:

            target = class_group_targets[class_id][split]
            current = class_group_current[split][class_id]

            if target > 0:
                score += max(
                    0,
                    target - current
                ) / target

        # Slight preference for filling the split proportionally
        capacity_ratio = (
            target_groups[split]
            - split_group_counts[split]
        ) / target_groups[split]

        score += 0.01 * capacity_ratio

        scores[split] = score

    # Deterministic tie breaking
    best_score = max(scores.values())

    best_splits = [
        split
        for split, score in scores.items()
        if np.isclose(score, best_score)
    ]

    chosen_split = sorted(best_splits)[0]

    assignments[group_id] = chosen_split

    split_group_counts[chosen_split] += 1

    for class_id in classes:
        class_group_current[chosen_split][class_id] += 1

# ------------------------------------------------------------
# Add split column to manifest
# ------------------------------------------------------------

df["split"] = df[GROUP_COLUMN].map(assignments)

if df["split"].isna().any():
    raise RuntimeError(
        "Some images were not assigned to a split."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("ACTUAL GROUP COUNTS")
print("-" * 40)

for split in ["train", "val", "test"]:
    print(
        f"{split:<10}: "
        f"{(df[GROUP_COLUMN][df['split'] == split].nunique()):,}"
    )

print()
print("ACTUAL IMAGE COUNTS")
print("-" * 40)

for split in ["train", "val", "test"]:
    print(
        f"{split:<10}: "
        f"{(df['split'] == split).sum():,}"
    )

CONSTRUCTING LEAKAGE-AWARE TRAIN / VAL / TEST SPLIT
Total groups : 6,439

Target groups:
  train     : 5,151
  val       : 644
  test      : 644

ACTUAL GROUP COUNTS
----------------------------------------
train     : 5,151
val       : 644
test      : 644

ACTUAL IMAGE COUNTS
----------------------------------------
train     : 5,326
val       : 668
test      : 662


In [11]:
# ============================================================
# Split integrity and class coverage audit
# ============================================================

print("=" * 70)
print("SPLIT INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check every image has exactly one split
# ------------------------------------------------------------

print("\n1. IMAGE ASSIGNMENT")
print("-" * 50)

print(f"Total images          : {len(df):,}")
print(f"Images with split     : {df['split'].notna().sum():,}")
print(f"Missing split         : {df['split'].isna().sum():,}")

assert df["split"].notna().all()

# ------------------------------------------------------------
# 2. Check duplicate image IDs
# ------------------------------------------------------------

print("\n2. IMAGE UNIQUENESS")
print("-" * 50)

duplicate_images = df["image_id"].duplicated().sum()

print(f"Duplicate image IDs   : {duplicate_images}")

assert duplicate_images == 0

# ------------------------------------------------------------
# 3. CRITICAL: Check group leakage
# ------------------------------------------------------------

print("\n3. NEAR-DUPLICATE GROUP LEAKAGE")
print("-" * 50)

group_split_counts = (
    df.groupby(GROUP_COLUMN)["split"]
      .nunique()
)

leaked_groups = group_split_counts[
    group_split_counts > 1
]

print(f"Total groups          : {len(group_split_counts):,}")
print(f"Groups crossing splits: {len(leaked_groups):,}")

if len(leaked_groups) == 0:
    print("✓ NO GROUP LEAKAGE")
else:
    print("✗ GROUP LEAKAGE DETECTED")
    display(leaked_groups.head(20))

assert len(leaked_groups) == 0

# ------------------------------------------------------------
# 4. Check split counts
# ------------------------------------------------------------

print("\n4. SPLIT COUNTS")
print("-" * 50)

split_summary = []

for split in ["train", "val", "test"]:

    subset = df[df["split"] == split]

    split_summary.append({
        "split": split,
        "images": len(subset),
        "groups": subset[GROUP_COLUMN].nunique()
    })

split_summary = pd.DataFrame(split_summary)

display(split_summary)

# ------------------------------------------------------------
# 5. Class coverage by split
# ------------------------------------------------------------

print("\n5. CLASS COVERAGE BY SPLIT")
print("-" * 50)

class_split_table = []

for class_id in sorted(class_names):

    row = {"class_id": class_id}

    for split in ["train", "val", "test"]:

        subset = df[df["split"] == split]

        count = sum(
            class_id in image_classes[image_id]
            for image_id in subset["image_id"]
        )

        row[split] = count

    class_split_table.append(row)

class_split_table = pd.DataFrame(class_split_table)

display(class_split_table)

# ------------------------------------------------------------
# 6. Identify missing classes
# ------------------------------------------------------------

print("\n6. MISSING CLASS COVERAGE")
print("-" * 50)

for split in ["train", "val", "test"]:

    missing = class_split_table[
        class_split_table[split] == 0
    ]["class_id"].tolist()

    if missing:
        print(f"{split:<10}: MISSING {missing}")
    else:
        print(f"{split:<10}: ✓ All 15 classes present")

# ------------------------------------------------------------
# 7. Class proportions
# ------------------------------------------------------------

print("\n7. CLASS PROPORTIONS")
print("-" * 50)

proportion_table = class_split_table.copy()

for split in ["train", "val", "test"]:

    total = proportion_table[split].sum()

    proportion_table[split] = (
        proportion_table[split] / total * 100
    )

display(
    proportion_table.round(2)
)

SPLIT INTEGRITY AUDIT

1. IMAGE ASSIGNMENT
--------------------------------------------------
Total images          : 6,656
Images with split     : 6,656
Missing split         : 0

2. IMAGE UNIQUENESS
--------------------------------------------------
Duplicate image IDs   : 0

3. NEAR-DUPLICATE GROUP LEAKAGE
--------------------------------------------------
Total groups          : 6,439
Groups crossing splits: 0
✓ NO GROUP LEAKAGE

4. SPLIT COUNTS
--------------------------------------------------


,split,images,groups
0,train,5326,5151
1,val,668,644
2,test,662,644



5. CLASS COVERAGE BY SPLIT
--------------------------------------------------


,class_id,train,val,test
0,0,2701,342,334
1,1,4042,513,504
2,2,4443,557,546
3,3,1081,136,139
4,4,811,104,106
5,5,872,111,108
6,6,138,18,17
7,7,147,19,20
8,8,490,64,61
9,9,578,71,72



6. MISSING CLASS COVERAGE
--------------------------------------------------
train     : ✓ All 15 classes present
val       : ✓ All 15 classes present
test      : ✓ All 15 classes present

7. CLASS PROPORTIONS
--------------------------------------------------


,class_id,train,val,test
0,0,16.74,16.74,16.57
1,1,25.05,25.11,25.00
2,2,27.54,27.26,27.08
3,3,6.70,6.66,6.89
4,4,5.03,5.09,5.26
5,5,5.41,5.43,5.36
6,6,0.86,0.88,0.84
7,7,0.91,0.93,0.99
8,8,3.04,3.13,3.03
9,9,3.58,3.48,3.57
